# Using `climate_toolkit` as a Python package

A tour of the main capabilities. Everything below is **credential-free** — it uses **NASA POWER** (gridded) and **NOAA GHCN** (weather stations), so **no Earth Engine / no API keys**. Run *Kernel → Restart & Run All*.

In [ ]:
%matplotlib inline
from datetime import date
import pandas as pd

import climate_toolkit as ct
from climate_toolkit.fetch_data.source_data.sources.utils.models import ClimateVariable

SITE = (-1.286, 36.817)   # Nairobi (lat, lon)
print("version:", ct.__version__)
[n for n in ct.__all__ if not n.startswith("__")]

## 1. Fetch daily climate data → a pandas DataFrame

*Module: data ingestion (`fetch_data`).*

In [ ]:
df = ct.fetch_climate_data(
    source="nasa_power",
    location_coord=SITE,
    variables=[ClimateVariable.precipitation,
               ClimateVariable.max_temperature,
               ClimateVariable.min_temperature],
    date_from=date(2020, 1, 1), date_to=date(2020, 3, 31),
    verbose=False,
)
print(df.shape)
df.head()

## 2. A quick visual

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
ax[0].plot(df["date"], df["max_temperature"], label="max T", color="tab:red")
ax[0].plot(df["date"], df["min_temperature"], label="min T", color="tab:blue")
ax[0].set_ylabel("°C"); ax[0].legend(loc="upper right")
ax[0].set_title("Nairobi daily climate — NASA POWER, 2020 Q1")
ax[1].bar(df["date"], df["precipitation"], color="tab:cyan")
ax[1].set_ylabel("precip (mm)")
fig.tight_layout(); plt.show()

## 3. Seasonal climatology & water balance

*Modules: `season_analysis` + `climate_statistics`.* Pin the March–May long-rains season and summarize each year (rainfall + water-balance indices NDWS/WRSI).

In [ ]:
stats = ct.analyze_climate_statistics(
    location_coord=SITE, start_year=2016, end_year=2020,
    source="nasa_power", fixed_season="03-01:05-31",
)
rows = [{
    "year": s["year"], "length_days": s["length_days"],
    "rain_mm": (s["precipitation"] or {}).get("total_mm"),
    "NDWS": (s["water_balance"] or {}).get("NDWS"),
    "WRSI": (s["water_balance"] or {}).get("WRSI"),
} for s in stats["season_statistics"]]
pd.DataFrame(rows)

## 4. Drought index — SPEI

*Module: `climatology`.*

In [ ]:
spei = ct.analyze_climate_statistics(
    location_coord=SITE, start_year=2016, end_year=2020,
    source="nasa_power", fixed_season="03-01:05-31", spei_scale_months=3,
)["spei"]
print("config:", spei["config"])
spei["summary"]

## 5. Weather-station observations

*Module: `weather_station`.* Auto-select and download the nearest quality NOAA GHCN station — also credential-free.

In [ ]:
station = ct.download_station_data(
    station_source="ghcn_daily", station_coord=SITE,
    date_from=date(2020, 1, 1), date_to=date(2020, 3, 31),
)
name = station["station_name"].iloc[0]
dist = station["station_distance_km"].iloc[0]
print(f"nearest station: {name}  ({dist:.1f} km away)  ->  {station.shape}")
station[["date", "station_name", "station_distance_km",
         "precipitation", "min_temperature"]].head()

## 6. Going further: Earth Engine sources & more

The **same functions** unlock gridded reanalysis and future projections after a one-time `earthengine authenticate` + `GCP_PROJECT_ID`. For reference (not run here — needs credentials):

```python
# Gridded reanalysis (ERA5-Land)
ct.fetch_climate_data(source="agera_5", location_coord=SITE,
    variables=[ClimateVariable.precipitation],
    date_from=date(2020, 1, 1), date_to=date(2020, 12, 31))

# Future projections (NEX-GDDP-CMIP6)
ct.fetch_climate_data(source="nex_gddp", model="GFDL-ESM4", scenario="ssp245",
    location_coord=SITE, variables=[ClimateVariable.precipitation],
    date_from=date(2050, 1, 1), date_to=date(2050, 12, 31))

# Compare datasets side by side
ct.compare_climate_sources(sources=["nasa_power", "agera_5"],
    lat=-1.286, lon=36.817, start="2020-01-01", end="2020-12-31")

# Validate a grid product against the nearby station
ct.compare_station_to_grids(station_source="ghcn_daily", station_coord=SITE,
    date_from=date(2020, 1, 1), date_to=date(2020, 3, 31),
    grid_sources=["nasa_power"])
```

## 7. Everything is discoverable

`help(ct.<function>)` (or `ct.fetch_climate_data?` in Jupyter) documents every parameter.

In [ ]:
print(ct.fetch_climate_data.__doc__[:500])